# M5.1 · Autoencoder ve VAE

Üretken modellere ilk adım. İki Gauss arasındaki KL'yi ve reparametrization gradyanlarını elle hesaplıyor, saf Python'da KL'nin kapalı formunu Monte Carlo tahminiyle doğruluyor, NumPy'da doğrusal bir autoencoder'ı elle türettiğin gradyanlarla eğitip PCA optimumuna indiğini görüyor, PyTorch'ta bir VAE yazıyor ve en sonda K örnekli IWAE sınırını tek bir batch'li hesapla, döngüsüz kuruyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m5-1-vae.html](https://m-gonul.github.io/pytorch-lab/m5-1-vae.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m5_1_vae.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    mu = [1.0, -0.5, 0.0]; lv = [math.log(0.25), 0.3, 0.0]\n    want = sum(0.5 * (m * m + math.exp(l) - 1 - l) for m, l in zip(mu, lv))\n    R.test(\"kl_gauss\", lambda: close(need(ns, \"kl_gauss\")(mu, lv), want))\n\n    def gr():\n        dmu, dlv = need(ns, \"kl_grad\")(mu, lv)\n        h = 1e-6\n        f = ns[\"kl_gauss\"]\n        num_mu = [(f(mu[:i] + [mu[i] + h] + mu[i + 1:], lv) - f(mu[:i] + [mu[i] - h] + mu[i + 1:], lv)) / (2 * h) for i in range(3)]\n        num_lv = [(f(mu, lv[:i] + [lv[i] + h] + lv[i + 1:]) - f(mu, lv[:i] + [lv[i] - h] + lv[i + 1:])) / (2 * h) for i in range(3)]\n        close(dmu, num_mu, atol=1e-5, what=\"dmu\"); close(dlv, num_lv, atol=1e-5, what=\"dlogvar\")\n    R.test(\"kl_grad = sayısal türev\", gr)\n    R.test(\"reparam\", lambda: close(need(ns, \"reparam\")([1.0, 2.0], [math.log(0.25), 0.0], [0.4, -1.0]), [1.2, 1.0]))\n    R.test(\"log_normal\", lambda: close(need(ns, \"log_normal\")([0.5, 1.0], [0.0, 2.0], [0.0, math.log(4)]),\n                                         -0.5 * (2 * math.log(2 * math.pi) + math.log(4) + 0.25 + 0.25)))\n\n    def mc():\n        rng = random.Random(0)\n        eps = [[rng.gauss(0, 1) for _ in range(3)] for _ in range(20000)]\n        est = need(ns, \"kl_mc\")(mu, lv, eps)\n        ensure(abs(est - want) < 0.05, f\"MC {est:.4f}, kapalı form {want:.4f}\")\n        return f\"20000 örnek: MC {est:.4f} ≈ kapalı form {want:.4f}\"\n    R.test(\"Monte Carlo KL → kapalı form\", mc)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"ae_grads döngüsüz\", lambda: max_loops(src, 0, func=\"ae_grads\"))\n    R.test(\"pca_error döngüsüz\", lambda: max_loops(src, 0, func=\"pca_error\"))\n    R.test(\"train_linear_ae: tek döngü\", lambda: max_loops(src, 1, func=\"train_linear_ae\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(7)\n    X = rng.normal(size=(50, 4)); We = rng.normal(size=(4, 2)); Wd = rng.normal(size=(2, 4))\n\n    def gr():\n        L, dWe, dWd = need(ns, \"ae_grads\")(X, We, Wd)\n        f = lambda We_, Wd_: ((X @ We_ @ Wd_ - X) ** 2).sum() / len(X)\n        close(L, f(We, Wd))\n        h = 1e-6\n        for W, dW, which in ((We, dWe, 0), (Wd, dWd, 1)):\n            num = np.zeros_like(W)\n            for idx in np.ndindex(W.shape):\n                Wp = W.copy(); Wp[idx] += h; Wm = W.copy(); Wm[idx] -= h\n                num[idx] = (f(Wp, Wd) - f(Wm, Wd)) / (2 * h) if which == 0 else (f(We, Wp) - f(We, Wm)) / (2 * h)\n            close(dW, num, atol=1e-4, what=\"dWe\" if which == 0 else \"dWd\")\n    R.test(\"ae_grads = sayısal türev\", gr)\n\n    def pe():\n        Y = X - X.mean(0)\n        U, s, Vt = np.linalg.svd(Y, full_matrices=False)\n        best = Y - (U[:, :2] * s[:2]) @ Vt[:2]\n        close(need(ns, \"pca_error\")(Y, 2), (best ** 2).sum() / len(Y))\n    R.test(\"pca_error = rank-2 SVD hatası\", pe)\n\n    def tr():\n        X = rng.normal(size=(200, 5)) * np.array([3, 2, 1, 0.3, 0.1])\n        X = X @ np.linalg.qr(rng.normal(size=(5, 5)))[0]; X -= X.mean(0)\n        We, Wd, L = need(ns, \"train_linear_ae\")(X, 2)\n        opt = need(ns, \"pca_error\")(X, 2)\n        ensure(L < opt * 1.01, f\"AE hatası {L:.4f}, PCA optimumu {opt:.4f}\")\n        P = Wd.T @ np.linalg.pinv(Wd.T)\n        _, _, Vt = np.linalg.svd(X, full_matrices=False)\n        Q = Vt[:2].T @ Vt[:2]\n        ensure(np.abs(P - Q).max() < 0.05, \"çözücünün satır uzayı ilk 2 PCA yönüyle örtüşmeli\")\n        return f\"AE {L:.4f} ≈ PCA {opt:.4f}; aynı alt uzay\"\n    R.test(\"doğrusal AE = PCA\", tr)\n",
 "d": "import math\nimport torch\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    g = torch.Generator().manual_seed(1)\n\n    def lossf():\n        x, xh, mu, lv = (torch.randn(5, k, generator=g) for k in (4, 4, 2, 2))\n        want = (((x - xh) ** 2).sum(-1) + 0.5 * 0.5 * (mu ** 2 + lv.exp() - 1 - lv).sum(-1)).mean()\n        close(need(ns, \"elbo_loss\")(x, xh, mu, lv, beta=0.5), want)\n    R.test(\"elbo_loss formülü (β = 0.5)\", lossf)\n\n    def shapes():\n        torch.manual_seed(0)\n        m = need(ns, \"VAE\", \"sınıf\")(12, 3)\n        x = torch.randn(7, 12)\n        mu, lv = m.encode(x)\n        ensure(tuple(mu.shape) == (7, 3) and tuple(lv.shape) == (7, 3), \"encode → (B, d_lat) iki tensör\")\n        xh, mu2, _ = m(x)\n        ensure(tuple(xh.shape) == (7, 12), \"forward → xhat (B, d_in)\")\n        ensure(tuple(need(ns, \"sample\")(m, 4).shape) == (4, 12), \"sample(model, 4) → (4, d_in)\")\n    R.test(\"şekiller\", shapes)\n\n    def reparam():\n        torch.manual_seed(0)\n        m = need(ns, \"VAE\", \"sınıf\")(12, 3)\n        x = torch.randn(7, 12)\n        torch.manual_seed(5); a, _, _ = m(x)\n        torch.manual_seed(6); b, _, _ = m(x)\n        ensure(not torch.allclose(a, b), \"forward örneklemeli (rastgele) olmalı\")\n        a.sum().backward()\n        enc_params = [p for n, p in m.named_parameters() if p.grad is not None and p.grad.abs().sum() > 0]\n        ensure(len(enc_params) == len(list(m.parameters())), \"tüm parametreler (kodlayıcı dahil) yeniden yapılandırmadan gradyan almalı\")\n    R.test(\"reparametrization: gradyan kodlayıcıya akıyor\", reparam)\n\n    def train():\n        A = torch.randn(2, 12, generator=g)\n        X = torch.tanh(torch.randn(1000, 2, generator=g) @ A) + 0.05 * torch.randn(1000, 12, generator=g)\n        torch.manual_seed(0)\n        m = need(ns, \"VAE\", \"sınıf\")(12, 2)\n        opt = torch.optim.Adam(m.parameters(), lr=1e-2)\n        for step in range(1000):\n            loss = ns[\"elbo_loss\"](X, *m(X))\n            if step == 0:\n                first = loss.item()\n            opt.zero_grad(); loss.backward(); opt.step()\n        with torch.no_grad():\n            mu, lv = m.encode(X)\n            rec = ((X - m.decode(mu)) ** 2).sum(-1).mean().item()\n            kl = (0.5 * (mu ** 2 + lv.exp() - 1 - lv).sum(-1)).mean().item()\n        var = X.var(0).sum().item()\n        ensure(loss.item() < 0.6 * first, f\"kayıp {first:.2f} → {loss.item():.2f}\")\n        ensure(rec < 0.3 * var, f\"yeniden yapılandırma {rec:.3f}, veri varyansı {var:.3f}\")\n        ensure(kl > 0.5, f\"KL {kl:.3f}: gizil uzay kullanılmıyor (posterior collapse)\")\n        return f\"kayıp {first:.2f} → {loss.item():.2f}; rec {rec:.3f} / var {var:.3f}; KL {kl:.2f} nat\"\n    R.test(\"sentetik veride eğitim\", train)\n",
 "e": "import math\nimport torch\nfrom torch.distributions import Normal\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef ref(encode, decode, x, eps):\n    mu, lv = encode(x)\n    q = Normal(mu, torch.exp(0.5 * lv))\n    vals = []\n    for k in range(eps.shape[0]):\n        z = mu + torch.exp(0.5 * lv) * eps[k]\n        lw = Normal(decode(z), 1.0).log_prob(x).sum(-1) + Normal(0.0, 1.0).log_prob(z).sum(-1) - q.log_prob(z).sum(-1)\n        vals.append(lw)\n    lw = torch.stack(vals)\n    return (torch.logsumexp(lw, 0) - math.log(len(vals))).mean()\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"iwae_bound döngüsüz\", lambda: max_loops(src, 0, func=\"iwae_bound\"))\n    torch.manual_seed(0)\n    enc = torch.nn.Sequential(torch.nn.Linear(6, 16), torch.nn.Tanh(), torch.nn.Linear(16, 6)).double()\n    dec = torch.nn.Sequential(torch.nn.Linear(3, 16), torch.nn.Tanh(), torch.nn.Linear(16, 6)).double()\n    encode = lambda x: enc(x).chunk(2, -1)\n    g = torch.Generator().manual_seed(0)\n    x = torch.randn(10, 6, generator=g, dtype=torch.float64)\n\n    def eq():\n        eps = torch.randn(7, 10, 3, generator=g, dtype=torch.float64)\n        close(need(ns, \"iwae_bound\")(encode, dec, x, eps), ref(encode, dec, x, eps))\n    R.test(\"döngülü başvuru (torch.distributions) ile aynı, K = 7\", eq)\n\n    def k1():\n        eps = torch.randn(1, 10, 3, generator=g, dtype=torch.float64)\n        mu, lv = encode(x)\n        z = mu + torch.exp(0.5 * lv) * eps[0]\n        rec = Normal(dec(z), 1.0).log_prob(x).sum(-1)\n        kl = 0.5 * (mu ** 2 + lv.exp() - 1 - lv).sum(-1)\n        lw = rec + Normal(0.0, 1.0).log_prob(z).sum(-1) - Normal(mu, torch.exp(0.5 * lv)).log_prob(z).sum(-1)\n        close(need(ns, \"iwae_bound\")(encode, dec, x, eps), lw.mean())\n    R.test(\"K = 1 → tek örnekli ELBO\", k1)\n\n    def mono():\n        vals = []\n        with torch.no_grad():\n            for K in (1, 5, 50, 500):\n                vals.append(sum(ns[\"iwae_bound\"](encode, dec, x, torch.randn(K, 10, 3, generator=g, dtype=torch.float64)).item() for _ in range(20)) / 20)\n        need(ns, \"iwae_bound\")\n        ensure(all(a < b for a, b in zip(vals, vals[1:])), f\"K ile artmalı: {[round(v, 4) for v in vals]}\")\n        return \"K = 1, 5, 50, 500 → \" + \", \".join(f\"{v:.4f}\" for v in vals)\n    R.test(\"sınır K ile sıkılaşıyor\", mono)\n\n    def grad():\n        eps = torch.randn(4, 10, 3, generator=g, dtype=torch.float64)\n        need(ns, \"iwae_bound\")(encode, dec, x, eps).backward()\n        ensure(all(p.grad is not None for p in enc.parameters()), \"kodlayıcı gradyan almalı\")\n    R.test(\"geri geçiş\", grad)\n    if \"iwae_bound\" not in ns:\n        return\n    enc32 = enc.float(); dec32 = dec.float()\n    encode32 = lambda x: enc32(x).chunk(2, -1)\n    xb = torch.randn(256, 6); eb = torch.randn(200, 256, 3)\n    with torch.no_grad():\n        t_loop = bench(lambda: ref(encode32, dec32, xb, eb), repeat=3)\n        t_vec = bench(lambda: ns[\"iwae_bound\"](encode32, dec32, xb, eb), repeat=3)\n    R.table(\"K = 200, B = 256\", [\"yöntem\", \"süre\"],\n            [[\"K üzerinde döngü\", fmt_time(t_loop)], [\"senin (K·B tek batch)\", fmt_time(t_vec)]],\n            note=\"K örneği batch boyutuna katlamak: decoder bir kez, büyük bir matmul ile çalışır.\")\n",
 "x1": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        mu, sig = rng.normal(size=5), rng.random(5) + 0.3\n        kl = np.sum(np.log(1 / sig) + (sig ** 2 + mu ** 2) / 2 - 0.5)\n        close(need(ns, \"latent_loss\")(mu, np.log(sig ** 2)), kl, what=\"Géron formülü = KL(N(μ, σ²) ‖ N(0, 1))\")\n        x, a, s2 = 1.7, 2.0, 0.5\n        m, v = need(ns, \"true_posterior\")(x, a, s2)\n        close(float(need(ns, \"elbo\")(x, a, s2, m, v)), need(ns, \"log_evidence\")(x, a, s2), atol=1e-10, what=\"gerçek posteriorda ELBO = log p(x)\")\n        rows = []\n        for mq, vq in ((m, v), (m + 0.3, v), (m, 3 * v), (0.0, 1.0)):\n            e = float(ns[\"elbo\"](x, a, s2, mq, vq))\n            klq = 0.5 * (np.log(v / vq) + (vq + (mq - m) ** 2) / v - 1)\n            close(ns[\"log_evidence\"](x, a, s2) - e, klq, atol=1e-10, what=\"boşluk = KL(q ‖ p(z|x))\")\n            rows.append([f\"{mq:.3f}\", f\"{vq:.3f}\", f\"{e:.4f}\", f\"{klq:.4f}\"])\n        R_.table(f\"p(z) = N(0, 1), p(x|z) = N(2z, 0.5), x = 1.7: log p(x) = {ns['log_evidence'](x, a, s2):.4f}\", [\"q ortalaması\", \"q varyansı\", \"ELBO\", \"log p(x) − ELBO\"], rows,\n                 note=\"ELBO her q için log p(x)'in alt sınırıdır; boşluk tam olarak q ile gerçek posterior arasındaki KL'dir. ELBO'yu q'ya göre büyütmek posterior yaklaşımını iyileştirmek, kod çözücüye göre büyütmek modelin olabilirliğini artırmak demektir. Son satır: q önsele eşitse (kodlayıcı girdiyi yok sayıyor) KL terimi 0 ama boşluk büyük.\")\n    R_.test(\"gizli kayıp ve ELBO\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        eigs = np.array([8.0, 4.0, 2.0, 1.0, 0.5, 0.25])\n        ensure(need(ns, \"active_dims\")(eigs, 1.0, 1.0) == 3 and ns[\"active_dims\"](eigs, 1.0, 0.1) == 6, \"aktif boyutlar\")\n        close(need(ns, \"decoder_norms\")(eigs, 1.0, 1.0), np.sqrt(np.maximum(0, eigs - 1)), what=\"kod çözücü normları\")\n        rng = np.random.default_rng(0)\n        d, n, s2 = 6, 200000, 1.0\n        Wt = np.diag(np.sqrt(np.maximum(eigs - s2, 0)))\n        Z = rng.normal(size=(n, d))\n        X = Z @ Wt.T + rng.normal(0, np.sqrt(s2), (n, d))\n        lam = np.linalg.eigvalsh(np.cov(X, rowvar=False))[::-1]\n        rows = []\n        for beta in (0.1, 0.5, 1.0, 2.0, 4.0, 10.0):\n            kl = need(ns, \"kl_per_dim\")(lam, s2, beta)\n            rows.append([beta, ns[\"active_dims\"](lam, s2, beta), f\"{kl.sum():.3f}\", \" \".join(f\"{v:.2f}\" for v in ns[\"decoder_norms\"](lam, s2, beta))])\n        ensure(rows[0][1] >= 5 and rows[-1][1] == 0, \"β büyüdükçe tüm boyutlar çöker\")\n        R_.table(\"Doğrusal VAE optimumu, gözlem gürültüsü σ² = 1, veri özdeğerleri ≈ 8, 4, 2, 1, 1, 1\", [\"β\", \"aktif boyut\", \"toplam KL (nat)\", \"kod çözücü sütun normları\"], rows,\n                 note=\"KL terimi bir 'bilgi bütçesi'dir: bir gizli boyutu kullanmanın bedeli, yeniden yapılandırmada sağladığı kazancı (λ_i − βσ²) aşarsa o boyut çöker — posterior önsele eşitlenir. β-VAE (β > 1) bu yüzden daha az ve daha 'ayrık' boyut kullanır; güçlü (otoregresif) kod çözücülerde tüm boyutların çökmesi (posterior çöküşü) yaygındır, KL ısınması (β'yı 0'dan 1'e artırmak) ya da free bits bunu hafifletir.\")\n    R_.test(\"β-VAE ve çöküş\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"lerp\", \"slerp\", \"typicality\")])\n\n    def t():\n        rng = np.random.default_rng(0)\n        z0, z1 = rng.normal(size=512), rng.normal(size=512)\n        close(need(ns, \"slerp\")(z0, z1, [0, 1]), np.stack([z0, z1]), atol=1e-9, what=\"uç noktalar\")\n        close(need(ns, \"lerp\")(z0, z1, [0.5])[0], (z0 + z1) / 2, what=\"lerp\")\n        ts = np.linspace(0, 1, 5)\n        L, S = ns[\"lerp\"](z0, z1, ts), ns[\"slerp\"](z0, z1, ts)\n        tl, tsl = need(ns, \"typicality\")(L), ns[\"typicality\"](S)\n        ensure(tl[2] < 0.8 and abs(tsl[2] - 1) < 0.05, \"lerp orta noktası atipik, slerp değil\")\n        rows = []\n        for d in (2, 16, 128, 1024):\n            a, b = rng.normal(size=d), rng.normal(size=d)\n            rows.append([d, f\"{float(ns['typicality'](ns['lerp'](a, b, [0.5]))[0]):.3f}\", f\"{float(ns['typicality'](ns['slerp'](a, b, [0.5]))[0]):.3f}\"])\n        R_.table(\"İki standart normal kodun orta noktası: norm/√d (önselden çekilmiş tipik kod ≈ 1)\", [\"d\", \"lerp\", \"slerp\"], rows,\n                 note=\"Bağımsız iki Gauss vektörü yüksek boyutta neredeyse diktir; ortalamalarının normu ≈ √(d/2): kodların ~%71'i büyüklüğünde, önselde neredeyse hiç görülmeyen bir bölgede. VAE'nin KL terimi kodları N(0, I)'ye çektiği için kod çözücü kabuk üzerinde eğitilmiştir — slerp, interpolasyon boyunca 'gerçekçi' kodlar verir (White 2016; GAN gizli uzaylarında da standart).\")\n    R_.test(\"interpolasyon\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m5_1_vae as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Gizil değişkenli model: $z\sim p(z)=\mathcal N(0,I)$, $x\sim p_\theta(x\mid z)$. $\log p_\theta(x)$ hesaplanamaz, ama bir kodlayıcı $q_\phi(z\mid x)=\mathcal N(\mu_\phi(x),\operatorname{diag}\sigma^2_\phi(x))$ ile alt sınır (ELBO) kurulur:

$$
\log p_\theta(x)\ \ge\ \mathbb E_{q_\phi}\big[\log p_\theta(x\mid z)\big]-\mathrm{KL}\big(q_\phi(z\mid x)\,\|\,p(z)\big).
$$

Gauss için KL kapalı formda: $\ \mathrm{KL}=\tfrac12\sum_j\big(\mu_j^2+\sigma_j^2-1-\log\sigma_j^2\big)$.

Beklentiyi tek örnekle tahmin ederken reparametrization trick: $z=\mu+\sigma\odot\varepsilon$, $\varepsilon\sim\mathcal N(0,I)$. Ağlar genelde $\log\sigma^2$ (`logvar`) üretir; böylece $\sigma>0$ kısıtı kendiliğinden sağlanır.

**iwae:** $K$ örnekle $\ \log\frac1K\sum_k\frac{p(x,z_k)}{q(z_k\mid x)}$; $K=1$ ELBO'dur, $K\to\infty$ için $\log p(x)$'e yaklaşır.

## 1 · Elle

<!-- çözüm -->

$\sigma$ küçükken $-\log\sigma^2$ terimi KL'yi büyütür: kodlayıcının çok emin (dar) olması cezalandırılır. $\mu$ sıfırdan uzaklaştıkça da $\mu^2/2$ cezası gelir. İkisi birlikte gizil uzayı $\mathcal N(0,I)$ etrafında tutar; prior'dan örnekleyip decode edebilmenin sebebi bu.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$q=\mathcal N(\mu,\sigma^2)$, $p=\mathcal N(0,1)$, $\mu=1$, $\sigma=0.5$**

- `kl`: $\mathrm{KL}(q\,\|\,p)=\tfrac12(\mu^2+\sigma^2-1-\log\sigma^2)$
- `dmu`: $\partial\mathrm{KL}/\partial\mu$
- `dsig`: $\partial\mathrm{KL}/\partial\sigma$

**Reparametrization: $z=\mu+\sigma\varepsilon$, $\varepsilon=0.4$**

- `z`: $z$
- `dzds`: $\partial z/\partial\sigma$
- `dlv`: $\log\sigma^2$ parametrizasyonunda $\partial\mathrm{KL}/\partial(\log\sigma^2)$

In [ ]:
cevaplar = dict(
    kl=None,
    dmu=None,
    dsig=None,
    z=None,
    dzds=None,
    dlv=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"kl":[0.818147,0.0005],"dmu":[1,0],"dsig":[-1.5,0],"z":[1.2,0],"dzds":[0.4,0],"dlv":[-0.375,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python; `mu`, `logvar` aynı uzunlukta listeler (diagonal Gauss).

- `kl_gauss(mu, logvar)` — kapalı form, boyutlar üzerinde toplam.
- `kl_grad(mu, logvar)` → `(dmu, dlogvar)` listeleri.
- `reparam(mu, logvar, eps)` → $z$.
- `log_normal(z, mu, logvar)` — diagonal Gauss log yoğunluğu.
- `kl_mc(mu, logvar, eps_list)` — Monte Carlo: her $\varepsilon$ için $z$, sonra $\log q(z)-\log p(z)$'nin ortalaması.

Kontrol, çok örnekli MC tahminin kapalı forma yaklaştığını doğrular.

In [ ]:
%%writefile stage_b.py
import math


def kl_gauss(mu, logvar):
    ...


def kl_grad(mu, logvar):
    ...


def reparam(mu, logvar, eps):
    ...


def log_normal(z, mu, logvar):
    ...


def kl_mc(mu, logvar, eps_list):
    ...


print(kl_gauss([1.0], [math.log(0.25)]))    # 0.818
print(reparam([1.0], [math.log(0.25)], [0.4]))   # [1.2]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

Doğrusal autoencoder: $Z=XW_e$, $\hat X=ZW_d$, $L=\frac1n\lVert\hat X-X\rVert_F^2$. Gradyanları kendin yaz:

$$
G=\tfrac2n(\hat X-X),\quad \nabla_{W_d}=Z^\top G,\quad \nabla_{W_e}=X^\top G\,W_d^\top.
$$

- `ae_grads(X, We, Wd)` → `(L, dWe, dWd)`. Döngü yok.
- `train_linear_ae(X, k, steps=3000, lr=0.02, seed=0)` → `(We, Wd, L)`; başlangıç `0.1 * normal`. Tek döngü (adımlar).
- `pca_error(X, k)` — en iyi rank-$k$ yeniden yapılandırmanın hatası: $X^\top X/n$'nin en küçük $d-k$ özdeğerinin toplamı.

Kontrol, eğitilen autoencoder'ın PCA optimumuna %1 içinde indiğini sınar.

In [ ]:
%%writefile stage_c.py
import numpy as np


def ae_grads(X, We, Wd):
    ...


def train_linear_ae(X, k, steps=3000, lr=0.02, seed=0):
    ...


def pca_error(X, k):
    ...


rng = np.random.default_rng(0)
X = rng.normal(size=(200, 5)) * np.array([3, 2, 1, 0.3, 0.1])
X = X @ np.linalg.qr(rng.normal(size=(5, 5)))[0]
X -= X.mean(0)
We, Wd, L = train_linear_ae(X, 2)
print(L, pca_error(X, 2))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `VAE(d_in, d_lat, hidden)` — `encode(x)` → `(mu, logvar)`; `decode(z)` → $\hat x$; `forward(x)` → `(xhat, mu, logvar)` (reparametrization trick ile). Kodlayıcı ve çözücü: `Linear → ReLU → Linear`.
- `elbo_loss(x, xhat, mu, logvar, beta=1.0)` — örnek başına $\lVert x-\hat x\rVert^2 + \beta\cdot\mathrm{KL}$, batch ortalaması.
- `sample(model, n)` — prior'dan $z$, decode.

Kontrol modeli 12 boyutlu, 2 gizil faktörlü sentetik veride eğitir: kayıp düşmeli, yeniden yapılandırma veri varyansının küçük bir kesrine inmeli ve KL sıfıra çökmemeli.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F
from torch import nn


class VAE(nn.Module):
    def __init__(self, d_in, d_lat, hidden=64):
        super().__init__()
        ...

    def encode(self, x):
        ...

    def decode(self, z):
        ...

    def forward(self, x):
        ...


def elbo_loss(x, xhat, mu, logvar, beta=1.0):
    ...


@torch.no_grad()
def sample(model, n):
    ...


g = torch.Generator().manual_seed(0)
A = torch.randn(2, 12, generator=g)
X = torch.tanh(torch.randn(1000, 2, generator=g) @ A) + 0.05 * torch.randn(1000, 12, generator=g)
torch.manual_seed(0)
m = VAE(12, 2)
opt = torch.optim.Adam(m.parameters(), lr=1e-2)
for step in range(1000):
    loss = elbo_loss(X, *m(X))
    opt.zero_grad(); loss.backward(); opt.step()
print(loss.item(), sample(m, 3).shape)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`iwae_bound(encode, decode, x, eps)` — `eps: (K, B, L)`. Hepsi tek seferde:

1. `mu, logvar = encode(x)`; `z = mu + std * eps` → `(K, B, L)` (yayınlama).
2. `decode(z.reshape(K * B, L))` → `(K, B, d)`.
3. $\log w_k=\log p(x\mid z_k)+\log p(z_k)-\log q(z_k\mid x)$; $p(x\mid z)=\mathcal N(\hat x, I)$.
4. Sonuç: $\operatorname{mean}_b\big[\operatorname{logsumexp}_k\log w_k-\log K\big]$.

**Döngü yok** ($K$ üzerinde de). Kontrol, $K=1$'de ELBO'ya eşitliği, döngülü başvuruyla aynılığı ve sınırın $K$ ile arttığını sınar.

In [ ]:
%%writefile stage_e.py
import math
import torch


def iwae_bound(encode, decode, x, eps):
    """eps: (K, B, L) → skaler. Döngü yok."""
    ...


torch.manual_seed(0)
enc = torch.nn.Linear(6, 4); dec = torch.nn.Linear(2, 6)
encode = lambda x: enc(x).chunk(2, -1)
x = torch.randn(8, 6)
for K in (1, 10, 100):
    print(K, iwae_bound(encode, dec, x, torch.randn(K, 8, 2)).item())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Gizli kayıp (γ = log σ²) ve ELBO boşluğu

**Gizli kayıp ve γ = log σ² parametrizasyonu** (Géron): kodlayıcı σ yerine $\gamma=\log\sigma^2$ üretince $\text{latent loss}=-\tfrac12\sum_i(1+\gamma_i-e^{\gamma_i}-\mu_i^2)$ — σ'nın pozitifliği kendiliğinden sağlanır, log/kare alma yok: sayısal olarak kararlı. Bu, $\mathrm{KL}(N(\mu,\sigma^2)\Vert N(0,1))$'in tam kendisidir. **ELBO türetmesi** (ders notu): $\log p(x)=\log\mathbb E_q[p(x,z)/q(z|x)]\ge\mathbb E_q[\log p(x|z)]-\mathrm{KL}(q\Vert p)$ (Jensen); boşluk $\mathrm{KL}(q(z|x)\Vert p(z|x))$'tir.

NumPy. Doğrusal-Gauss oyuncak model: $p(z)=N(0,1)$, $p(x|z)=N(az,s^2)$ — gerçek posterior ve $\log p(x)$ kapalı formda.

- `latent_loss(mu, gamma)` — Géron'un formülü (son eksende toplam).
- `elbo(x, a, s2, m, v)` — $q(z|x)=N(m,v)$ için ELBO, kapalı form: $\mathbb E_q\log p(x|z)=-\tfrac12\log(2\pi s^2)-\frac{(x-am)^2+a^2v}{2s^2}$.
- `log_evidence(x, a, s2)` ve `true_posterior(x, a, s2)` → `(m*, v*)`.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def latent_loss(mu, gamma):
    ...


def elbo(x, a, s2, m, v):
    ...


def log_evidence(x, a, s2):
    ...


def true_posterior(x, a, s2):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · β-VAE ve posterior çöküşü: doğrusal VAE'nin kapalı form optimumu

**β-VAE ve posterior çöküşü** (Higgins ve ark. 2017; Lucas ve ark. 2019). Doğrusal bir VAE'de (doğrusal kodlayıcı/kod çözücü, Gauss gözlem gürültüsü σ²) amacın optimumu kapalı formdadır (olasılıksal PCA'nın genelleşmesi): veri kovaryansının özdeğeri $\lambda_i>\beta\sigma^2$ olan yönler gizli boyut olarak kullanılır, diğerleri **çöker** (posterior önsele eşit, kod çözücü o boyutu yok sayar). Optimum kod çözücü sütun normu $\sqrt{\lambda_i-\beta\sigma^2}$ (pozitifse).

NumPy.

- `active_dims(eigs, sigma2, beta)` → aktif boyut sayısı.
- `decoder_norms(eigs, sigma2, beta)` → $\sqrt{\max(0,\lambda_i-\beta\sigma^2)}$.
- `kl_per_dim(eigs, sigma2, beta)` — optimumda boyut başına KL: aktif boyutlarda $\tfrac12(\log(\lambda_i/\beta\sigma^2))$, çökmüşlerde 0.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def active_dims(eigs, sigma2, beta):
    ...


def decoder_norms(eigs, sigma2, beta):
    ...


def kl_per_dim(eigs, sigma2, beta):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Gizli uzayda interpolasyon: lerp vs slerp

**Gizli uzayda interpolasyon** (Géron'un "anlamsal interpolasyon"u): iki görüntüyü kodlayıcıdan geçir, kodlar arasında interpole et, kod çözücüden geçir. Yüksek boyutlu Gauss önselinde kütle yarıçap ≈ √d olan bir kabukta yoğunlaşır; doğrusal interpolasyonun orta noktası bu kabuğun içine düşer (norm ≈ √(d/2)) — kod çözücünün hiç görmediği "atipik" kodlar. Küresel doğrusal interpolasyon (slerp, White 2016) kabuk üzerinde kalır.

NumPy.

- `lerp(z0, z1, t)` ve `slerp(z0, z1, t)` — $t$ dizi olabilir; sonuç `(len(t), d)`.
- `typicality(Z)` — kodların normları / √d.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def lerp(z0, z1, t):
    ...


def slerp(z0, z1, t):
    ...


def typicality(Z):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Kingma & Welling (2014): Auto-Encoding Variational Bayes

[Kingma & Welling — Auto-Encoding Variational Bayes (ICLR 2014)](https://arxiv.org/abs/1312.6114)

Makalenin Şekil 2 deneyini tekrarla: MNIST/Frey Face'te gizli boyut N_z ∈ {3, 5, 10, 20} için d aşamasındaki VAE'yi eğitip ELBO'yu (alt sınır) eğitim örneği sayısına göre çiz; Şekil 4'teki 2B gizli uzay manifoldunu (ızgara üzerinde kod çözme) üret. Önemli örneklemeli tahminci (M5.1 e: IWAE) ile log p(x) tahminini ELBO ile karşılaştır.


**Adımlar**

1. Izgarayı Gauss önselin ters CDF'iyle oluştur (makalenin önerisi).

**Kabul ölçütleri**

- Daha büyük N_z aşırı uyum göstermeden ELBO'yu iyileştirir (makaleyle uyumlu).
- 2B manifold rakamlar arasında yumuşak geçişler gösterir.

## Makale ödevi: Higgins ve ark. (2017): β-VAE

[Higgins ve ark. — β-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework (ICLR 2017)](https://openreview.net/forum?id=Sy2fzU9gl)

dSprites veri setinde (şekil, ölçek, açı, x, y) β ∈ {1, 4, 16} ile konvolüsyonel VAE eğit; her gizli boyutu tek tek gezerek (latent traversal, Şekil 7) hangi üretici faktörü kodladığını göster ve x2'deki gibi aktif boyut sayısını (KL > 0.01 nat) β'ya göre raporla. Makalenin ayrıklık (disentanglement) ölçütünü basitleştirilmiş biçimde uygula.


**Adımlar**

1. KL ısınması (β'yı 0'dan hedefe doğrusal) ile ve onsuz çöken boyut sayısını karşılaştır.

**Kabul ölçütleri**

- β arttıkça daha az ama daha yorumlanabilir aktif boyut.
- Yeniden yapılandırma kalitesi–ayrıklık ödünleşimi tablosu.

## Kaynaklar

- [Kingma & Welling — Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) — arXiv
- [Burda, Grosse, Salakhutdinov — Importance Weighted Autoencoders](https://arxiv.org/abs/1509.00519) — arXiv
- [Prince — Understanding Deep Learning, Bölüm 17 (VAE)](https://udlbook.github.io/udlbook/) — kitap ©; notebook'lar MIT
- [UvA Deep Learning Tutorials — Autoencoders](https://uvadlc-notebooks.readthedocs.io/) — MIT
- [Stanford CS236 — Deep Generative Models](https://deepgenerativemodels.github.io/) — doğrulanmadı